# Yêu cầu 4: Model Deployment
Triển khai model dưới dạng REST API bằng FastAPI.
- Endpoint: `/predict`
- Input: text string
- Output: predicted label (`ham` / `spam`)
- Load model từ MLflow Registry (Production)

## 1. Code FastAPI (lưu vào file `app.py`)

In [ ]:
app_code = '''
import re
import joblib
import mlflow
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title='Spam Classifier API')

mlflow.set_tracking_uri("file:../2_ModelTraining/mlruns")
model = mlflow.pyfunc.load_model("models:/spam_classifier/Production")
vectorizer = joblib.load("../1_DataPipeline/processed/tfidf_vectorizer.pkl")

def clean_text(t):
    t = t.lower()
    t = re.sub(r"[^a-z\\s]", " ", t)
    t = re.sub(r"\\s+", " ", t).strip()
    return t

class Item(BaseModel):
    text: str

@app.get("/")
def root():
    return {"message": "Spam Classifier API is running"}

@app.post("/predict")
def predict(item: Item):
    x = vectorizer.transform([clean_text(item.text)])
    pred = int(model.predict(x)[0])
    label = "spam" if pred == 1 else "ham"
    return {"input": item.text, "predicted_label": label}
'''
with open('app.py', 'w') as f:
    f.write(app_code)
print('Đã tạo app.py')

## 2. Chạy server
Mở terminal trong thư mục `4_ModelDeployment` và chạy:
```bash
uvicorn app:app --reload --port 8000
```

## 3. Test bằng curl
```bash
curl -X POST http://127.0.0.1:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"text": "Free entry to win an iPhone, click here!"}'
```
**Kết quả mong đợi:**
```json
{"input": "Free entry to win an iPhone, click here!", "predicted_label": "spam"}
```

Hoặc dùng Postman:
- Method: POST
- URL: `http://127.0.0.1:8000/predict`
- Body (raw JSON): `{"text": "Hello, are you free tonight?"}`

## 4. Test trực tiếp bằng requests (tuỳ chọn)

In [ ]:
import requests
samples = [
    'Free entry to win an iPhone, click here!',
    'Hi mom, I will be home for dinner tonight.'
]
for s in samples:
    r = requests.post('http://127.0.0.1:8000/predict', json={'text': s})
    print(r.json())